In [1]:
import pandas as pd 
import os 
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [2]:
input_path = os.path.join("..", "data")
filepath = os.path.join(input_path, "disruptions_2024.csv")

df = pd.read_csv(filepath, sep=';') 

In [5]:
#Disruptions on Kingsday 2024
df['start_time'] = pd.to_datetime(df['start_time'],format='%d-%m-%Y %H:%M')

kingsday_df = df[df['start_time'].dt.date == pd.to_datetime('2024-04-27').date()]

overview_disruptions_df = kingsday_df[['start_time', 'ns_lines', 'cause_nl', 'duration_minutes']].sort_values(by='start_time')

overview_disruptions_df


,start_time,ns_lines,cause_nl,duration_minutes
1857,2024-04-27 10:40:00,Utrecht-Veenendaal,persoon op het spoor,8.0
1858,2024-04-27 11:22:00,Utrecht-Rotterdam,persoon op het spoor,38.0
1860,2024-04-27 12:14:00,Leeuwarden-Harlingen Haven,seinstoring,62.0
1861,2024-04-27 13:48:00,Amsterdam-Hilversum,aanrijding,10.0
1863,2024-04-27 15:03:00,Utrecht-Almere,aanrijding,86.0
1864,2024-04-27 15:03:00,Leiden-Den Haag,aanrijding,1.0
1865,2024-04-27 15:13:00,Leiden-Den Haag,aanrijding,77.0
1866,2024-04-27 15:33:00,Utrecht-'s-Hertogenbosch,defect spoor,21.0
1867,2024-04-27 20:26:00,Stavoren,defecte trein,0.0
1868,2024-04-27 20:29:00,Leeuwarden-Harlingen Haven,seinstoring,99.0


In [6]:
total_disruption_minutes = kingsday_df['duration_minutes'].sum()
print(f"The total disruption time for Kingsday 2024 is: {total_disruption_minutes} minutes.")        

The total disruption time for Kingsday 2024 is: 531.0 minutes.


In [7]:
# 23 of november has a high delay time, so we will filter the data for that day and see what happened 
df['start_time'] = pd.to_datetime(df['start_time'])


nov_df = df[df['start_time'].dt.date == pd.to_datetime('2024-11-23').date()]

overview_disruptions_df = nov_df[['start_time', 'ns_lines', 'cause_nl', 'duration_minutes']].sort_values(by='start_time')

overview_disruptions_df



,start_time,ns_lines,cause_nl,duration_minutes
5335,2024-11-23 07:24:00,Venlo - Mönchengladbach - Düsseldorf,problemen op het spoor in het buitenland,708.0
5337,2024-11-23 11:04:00,Zutphen - Hengelo,defecte trein,5.0
5338,2024-11-23 12:51:00,Schiphol - Rotterdam,gestrande trein,2.0
5339,2024-11-23 14:21:00,Rotterdam - Dordrecht,storing in een tunnel,157.0
5340,2024-11-23 14:32:00,Rotterdam - Breda,storing in een tunnel,163.0
5341,2024-11-23 14:36:00,Breda - Roosendaal,aanrijding,31.0
5343,2024-11-23 15:55:00,Hengelo - Bielefeld,defecte trein,13.0
5344,2024-11-23 16:27:00,Amersfoort - Zwolle,inzet van hulpdiensten,48.0
5345,2024-11-23 19:19:00,Arnhem - Zutphen,hinder op het spoor,56.0
5346,2024-11-23 19:54:00,Deventer - Almelo,defecte trein,37.0


In [8]:
total_disruption_minutes = nov_df['duration_minutes'].sum()
print(f"The total disruption time for 23 November 2024 is: {total_disruption_minutes} minutes.")

The total disruption time for 23 November 2024 is: 1468.0 minutes.


In [10]:
%pip install kaleido

Note: you may need to restart the kernel to use updated packages.


In [9]:
#compare the disruptions against the delays on saturdays in 2024, to see if there is a correlation between the two 
combined_df = pd.read_pickle("combined_df_zaterdagen.pkl")       #saved in previous notebook 

#The delay time per saturday 
train_saturdays = combined_df.groupby('Service:Date')['Stop:Arrival delay'].sum().reset_index()
train_saturdays.columns = ['Date', 'Minutes_Delay']
train_saturdays['Date'] = pd.to_datetime(train_saturdays['Date']).dt.date


#The disruption time per saturday 
saturdays = ['2024-01-06', '2024-01-13', '2024-01-20', '2024-01-27', '2024-02-03', '2024-02-10', '2024-02-17', '2024-02-24', '2024-03-02', '2024-03-09', '2024-03-16', '2024-03-23', '2024-03-30', '2024-04-06', '2024-04-13', '2024-04-20', '2024-04-27', '2024-05-04', '2024-05-11', '2024-05-18', '2024-05-25', '2024-06-01', '2024-06-08', '2024-06-15', '2024-06-22', '2024-06-29', '2024-07-06', '2024-07-13', '2024-07-20', '2024-07-27', '2024-08-03', '2024-08-10', '2024-08-17', '2024-08-24', '2024-08-31', '2024-09-07', '2024-09-14', '2024-09-21', '2024-09-28', '2024-10-05', '2024-10-12', '2024-10-19', '2024-10-26', '2024-11-02', '2024-11-09', '2024-11-16', '2024-11-23', '2024-11-30', '2024-12-07', '2024-12-14', '2024-12-21', '2024-12-28']
saturdays = pd.to_datetime(saturdays).date
df['Date'] = pd.to_datetime(df['start_time'], format='%d-%m-%Y %H:%M').dt.date
disruptions_2024 = df[df['Date'].isin(saturdays)]
disruptions_saturdays = disruptions_2024.groupby('Date')['duration_minutes'].sum().reset_index()
disruptions_saturdays.columns = ['Date', 'Minutes_Disruptions']

#Merge the two datasets together 
comparison_df = pd.merge(train_saturdays, disruptions_saturdays, on='Date').sort_values(by='Date')


#Plotly dataframe in long format 
comparison_long = pd.melt(
    comparison_df, 
    id_vars=['Date'], 
    value_vars=['Minutes_Delay', 'Minutes_Disruptions'],
    var_name='Category', 
    value_name='Minutes'
)
comparison_long['Date'] = comparison_long['Date'].astype(str)

#plotly bar chart
fig = px.bar(
    comparison_long,
    x='Date',
    y='Minutes',
    color='Category',
    barmode='group',
    labels={'Date': 'Date Saturday', 'Minutes': 'Total Time (in minutes)'},
    title='Total Delay vs. Total Disruption time (Saturdays 2024)',
)

# Highlight Kings Day 
highlight = '2024-04-27'   
orange = 'orange'

for trace in fig.data:
    base_color = trace.marker.color   
    trace.marker.color = [orange if i == highlight else base_color for i in trace.x]

os.makedirs(os.path.join("..", "Figures"), exist_ok=True)
fig.write_image(os.path.join("..", "Figures", "figure4.png"), scale=2)

fig.show()


